In [1]:
# %% 셀 1: 데이터 로드 (텔롭 bbox 예측 모델)
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
HEATMAP_DIR = "./data/8_heatmap"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SLOT_DIM = 5
TIME_DIM = 3

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel,
        "video_name": video_name,
        "file_id": file_id,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개")
print(f"   train: {len(train_samples):,}  eval: {len(eval_samples_with):,}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024


로드: 100%|██████████| 66400/66400 [00:14<00:00, 4638.28it/s]


✅ 채널: 664개
   train: 56,892  eval: 2,984


In [2]:
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples = [s for s in eval_samples if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

eval_samples_with = [s for s in eval_samples if len(s["instances"]) > 0]
train_samples = [s for s in train_samples if len(s["instances"]) > 0]

print(f"\n🔬 샘플링: {len(channels)}개 채널")
print(f"   train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")


🔬 샘플링: 67개 채널
   train: 5,574  |  eval: 335


In [3]:
# %% 셀 2: Dataset + DataLoader (bbox 예측, 프레임 단위, V9 히트맵)
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 2
NUM_WORKERS_DL = 4
HEATMAP_PATCH = 8
N_HMAP_PATCHES = (GRID_H // HEATMAP_PATCH) * (GRID_W // HEATMAP_PATCH)
CACHE_LIMIT = 50


class BboxFrameDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb
        self._cache = {}
        self._cache_order = []

    def _get_heatmap(self, channel, file_id, n_frames):
        if channel not in self._cache:
            if len(self._cache_order) >= CACHE_LIMIT:
                old_ch = self._cache_order.pop(0)
                del self._cache[old_ch]
            path = os.path.join(HEATMAP_DIR, f"{channel}.pt")
            if os.path.exists(path):
                self._cache[channel] = torch.load(path, weights_only=True)
            else:
                self._cache[channel] = {}
            self._cache_order.append(channel)

        heatmap = self._cache[channel].get(file_id)
        if heatmap is not None:
            hmap = heatmap.float().numpy()[:n_frames]
            if hmap.shape[0] < n_frames:
                pad = np.zeros((n_frames - hmap.shape[0], GRID_H, GRID_W), dtype=np.float32)
                hmap = np.concatenate([hmap, pad], axis=0)
            return hmap
        return np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        channel_id = self.channel2id[s["channel"]]
        instances = s["instances"]
        duration = max(s["duration"], 0.1)
        n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
        n_inst = len(instances)

        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends   = np.array([inst["end"]   for inst in instances])
        inst_tlens  = np.array([inst["text_len"] for inst in instances])
        inst_cx = np.array([inst["x"] for inst in instances])
        inst_cy = np.array([inst["y"] for inst in instances])
        inst_w  = np.array([inst["w"] for inst in instances])
        inst_h  = np.array([inst["h"] for inst in instances])

        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)
        video_emb   = self.text2emb.get(s["video_name"], ZERO_EMB)
        inst_embs = torch.stack([self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

        stt_sims = np.zeros(n_inst, dtype=np.float32)
        has_stts = np.zeros(n_inst, dtype=np.float32)
        stt_segments = s["stt_segments"]
        if len(stt_segments) > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends   = np.array([seg["end"]   for seg in stt_segments])
            stt_embs   = torch.stack([self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
            for i in range(n_inst):
                mid = (inst_starts[i] + inst_ends[i]) / 2
                stt_active = (stt_starts <= mid) & (stt_ends > mid)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_sims[i] = F.cosine_similarity(
                        inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                    has_stts[i] = 1.0

        times = np.arange(n_frames, dtype=np.float32) / FPS
        active_matrix = (inst_starts[None, :] <= times[:, None] + 0.05) & (inst_ends[None, :] > times[:, None])

        telop_feats = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, SLOT_DIM), dtype=np.float32)
        telop_mask  = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
        gt_bbox     = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.float32)

        for fi in range(n_frames):
            active_idx = np.where(active_matrix[fi])[0]
            if len(active_idx) > 0:
                sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
                sorted_idx = active_idx[sorted_order]
                for si, ai in enumerate(sorted_idx):
                    telop_feats[fi, si, 0] = inst_tlens[ai] / MAX_TEXT_LEN
                    telop_feats[fi, si, 1] = ch_sims[ai]
                    telop_feats[fi, si, 2] = vid_sims[ai]
                    telop_feats[fi, si, 3] = stt_sims[ai]
                    telop_feats[fi, si, 4] = has_stts[ai]
                    telop_mask[fi, si] = True

                    gt_bbox[fi, si, 0] = inst_cx[ai] / GRID_W
                    gt_bbox[fi, si, 1] = inst_cy[ai] / GRID_H
                    gt_bbox[fi, si, 2] = inst_w[ai]  / GRID_W
                    gt_bbox[fi, si, 3] = inst_h[ai]  / GRID_H

        heatmaps = self._get_heatmap(s["channel"], s["file_id"], n_frames)

        time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
        t_norm = times / max(duration, 1.0)
        time_feats[:, 0] = t_norm
        time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
        time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

        return {
            "channel_id":  torch.tensor(channel_id, dtype=torch.long),
            "telop_feats": torch.from_numpy(telop_feats),
            "telop_mask":  torch.from_numpy(telop_mask),
            "time_feats":  torch.from_numpy(time_feats),
            "heatmaps":    torch.from_numpy(heatmaps),
            "gt_bbox":     torch.from_numpy(gt_bbox),
            "n_frames":    n_frames,
        }


def collate_fn(batch):
    max_T = max(b["n_frames"] for b in batch)
    B = len(batch)

    channel_ids = torch.stack([b["channel_id"] for b in batch])
    telop_feats = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, SLOT_DIM)
    telop_mask  = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, dtype=torch.bool)
    time_feats  = torch.zeros(B, max_T, TIME_DIM)
    heatmaps    = torch.zeros(B, max_T, GRID_H, GRID_W)
    gt_bbox     = torch.zeros(B, max_T, MAX_ACTIVE_PER_FRAME, 4)
    frame_mask  = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["n_frames"]
        telop_feats[i, :T] = b["telop_feats"]
        telop_mask[i, :T]  = b["telop_mask"]
        time_feats[i, :T]  = b["time_feats"]
        heatmaps[i, :T]    = b["heatmaps"]
        gt_bbox[i, :T]     = b["gt_bbox"]
        frame_mask[i, :T]  = True

    return {
        "channel_ids": channel_ids,
        "telop_feats": telop_feats,
        "telop_mask":  telop_mask,
        "time_feats":  time_feats,
        "heatmaps":    heatmaps,
        "gt_bbox":     gt_bbox,
        "frame_mask":  frame_mask,
    }


train_ds = BboxFrameDataset(train_samples, channel2id, text2emb)
eval_ds  = BboxFrameDataset(eval_samples_with, channel2id, text2emb)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn, drop_last=True,
    persistent_workers=True,
)
eval_loader = DataLoader(
    eval_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS_DL, pin_memory=True,
    collate_fn=collate_fn,
    persistent_workers=True,
)

print(f"✅ Dataset: train={len(train_ds):,}  eval={len(eval_ds):,}")

batch = next(iter(train_loader))
print(f"\n✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

valid_bbox = batch["gt_bbox"][batch["telop_mask"]]
print(f"\n📊 활성 슬롯 bbox (정규화 0~1)")
for i, name in enumerate(["x", "y", "w", "h"]):
    vals = valid_bbox[:, i]
    print(f"  {name}: mean={vals.mean():.3f}  std={vals.std():.3f}  min={vals.min():.3f}  max={vals.max():.3f}")

✅ Dataset: train=5,574  eval=288

✅ 배치 확인
  channel_ids: torch.Size([2])
  telop_feats: torch.Size([2, 448, 150, 5])
  telop_mask: torch.Size([2, 448, 150])
  time_feats: torch.Size([2, 448, 3])
  heatmaps: torch.Size([2, 448, 80, 80])
  gt_bbox: torch.Size([2, 448, 150, 4])
  frame_mask: torch.Size([2, 448])

📊 활성 슬롯 bbox (정규화 0~1)
  x: mean=0.506  std=0.124  min=0.287  max=0.850
  y: mean=0.342  std=0.248  min=0.125  max=0.800
  w: mean=0.632  std=0.207  min=0.138  max=0.938
  h: mean=0.066  std=0.011  min=0.025  max=0.075


In [4]:
# %% 셀 3: 모델 (Heatmap-conditioned Bbox Predictor, 프레임 단위)
D_MODEL = 256
N_HEADS = 8
N_CROSS_LAYERS = 4
N_SELF_LAYERS = 2
D_FF = 512
DROPOUT = 0.1
HEATMAP_PATCH = 8
N_HMAP_PATCHES = (GRID_H // HEATMAP_PATCH) * (GRID_W // HEATMAP_PATCH)
FRAME_CHUNK = 64


class HeatmapBboxModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL, n_heads=N_HEADS,
                 n_cross_layers=N_CROSS_LAYERS, n_self_layers=N_SELF_LAYERS,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        # 히트맵 인코더
        self.hmap_patch_proj = nn.Linear(HEATMAP_PATCH * HEATMAP_PATCH, d_model)
        self.hmap_pos_emb = nn.Parameter(torch.randn(1, N_HMAP_PATCHES, d_model) * 0.02)
        self.hmap_norm = nn.LayerNorm(d_model)

        # 슬롯 인코더
        self.slot_proj = nn.Sequential(
            nn.Linear(SLOT_DIM, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        # 채널 + 시간
        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64), nn.GELU(), nn.Linear(64, d_model))

        # Cross-attention: K개 슬롯(query) ← 100 히트맵 패치(key/value)
        cross_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.cross_attn = nn.TransformerDecoder(
            cross_layer, num_layers=n_cross_layers)

        # Self-attention: K개 슬롯 간 겹침 방지
        self_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.self_attn = nn.TransformerEncoder(
            self_layer, num_layers=n_self_layers, enable_nested_tensor=False)

        # Bbox head
        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, 4), nn.Sigmoid())

    def encode_heatmap(self, heatmaps):
        N = heatmaps.shape[0]
        patches = (
            heatmaps
            .reshape(N, GRID_H // HEATMAP_PATCH, HEATMAP_PATCH,
                        GRID_W // HEATMAP_PATCH, HEATMAP_PATCH)
            .permute(0, 1, 3, 2, 4)
            .reshape(N, N_HMAP_PATCHES, HEATMAP_PATCH * HEATMAP_PATCH)
        )
        tokens = self.hmap_patch_proj(patches) + self.hmap_pos_emb
        return self.hmap_norm(tokens)

    def forward(self, channel_ids, telop_feats, telop_mask, time_feats,
                heatmaps, frame_mask):
        B, T, K, _ = telop_feats.shape
        device = telop_feats.device
        dtype = telop_feats.dtype
        BT = B * T

        valid_flat = frame_mask.reshape(BT)
        valid_idx = valid_flat.nonzero(as_tuple=True)[0]
        n_valid = valid_idx.shape[0]

        all_pred = torch.zeros(BT, K, 4, device=device, dtype=dtype)

        if n_valid == 0:
            return all_pred.reshape(B, T, K, 4)

        slot_flat  = telop_feats.reshape(BT, K, SLOT_DIM)[valid_idx]
        smask_flat = telop_mask.reshape(BT, K)[valid_idx]
        hmap_flat  = heatmaps.reshape(BT, GRID_H, GRID_W)[valid_idx]

        ch_emb = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        ch_flat = ch_emb.reshape(BT, -1)[valid_idx]
        time_flat = self.time_proj(time_feats.reshape(BT, TIME_DIM)[valid_idx])

        chunk_preds = []
        for start in range(0, n_valid, FRAME_CHUNK):
            end = min(start + FRAME_CHUNK, n_valid)
            c_slots = slot_flat[start:end]
            c_smask = smask_flat[start:end]
            c_hmap  = hmap_flat[start:end]
            c_ch    = ch_flat[start:end]
            c_time  = time_flat[start:end]

            hmap_tokens = self.encode_heatmap(c_hmap)

            slot_tokens = self.slot_proj(c_slots)
            slot_tokens = slot_tokens + c_ch.unsqueeze(1) + c_time.unsqueeze(1)

            slot_pad = ~c_smask
            cross_out = self.cross_attn(
                tgt=slot_tokens, memory=hmap_tokens,
                tgt_key_padding_mask=slot_pad)

            self_out = self.self_attn(
                cross_out, src_key_padding_mask=slot_pad)

            pred = self.bbox_head(self_out)
            chunk_preds.append(pred)

        valid_pred = torch.cat(chunk_preds, dim=0)
        all_pred[valid_idx] = valid_pred.to(all_pred.dtype)

        return all_pred.reshape(B, T, K, 4)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HeatmapBboxModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")
print(f"   cross_attn: {N_CROSS_LAYERS} layers  self_attn: {N_SELF_LAYERS} layers")
print(f"   d_model={D_MODEL}  n_heads={N_HEADS}  d_ff={D_FF}")
print(f"   FRAME_CHUNK={FRAME_CHUNK}")

🖥️  Device: cuda
📐 파라미터: 4,428,292
   cross_attn: 4 layers  self_attn: 2 layers
   d_model=256  n_heads=8  d_ff=512
   FRAME_CHUNK=64


In [5]:
# %% 셀 4: 학습 (Heatmap-conditioned Bbox Predictor, 프레임 단위)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 100
LR = 1e-4
SAVE_DIR = "./model/8_telop_bbox_frame"
os.makedirs(SAVE_DIR, exist_ok=True)


@torch.no_grad()
def compute_bbox_metrics(pred_bbox, gt_bbox, telop_mask, frame_mask):
    valid_frames = frame_mask.unsqueeze(-1).expand_as(telop_mask)
    active = telop_mask & valid_frames

    if not active.any():
        return None

    pred = pred_bbox[active]
    gt   = gt_bbox[active]

    scale = torch.tensor([GRID_W, GRID_H, GRID_W, GRID_H],
                         device=pred.device, dtype=pred.dtype)
    pred_grid = pred * scale
    gt_grid   = gt * scale

    mae_per_dim = (pred_grid - gt_grid).abs().mean(dim=0)

    return {
        "mae_x": mae_per_dim[0].item(),
        "mae_y": mae_per_dim[1].item(),
        "mae_w": mae_per_dim[2].item(),
        "mae_h": mae_per_dim[3].item(),
        "mae_total": mae_per_dim.sum().item(),
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0.0, 0

    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_ids = batch["channel_ids"].to(device)
        telop_feats = batch["telop_feats"].to(device)
        telop_mask  = batch["telop_mask"].to(device)
        time_feats  = batch["time_feats"].to(device)
        heatmaps    = batch["heatmaps"].to(device)
        gt_bbox     = batch["gt_bbox"].to(device)
        frame_mask  = batch["frame_mask"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred_bbox = model(channel_ids, telop_feats, telop_mask,
                              time_feats, heatmaps, frame_mask)

            valid_frames = frame_mask.unsqueeze(-1).expand_as(telop_mask)
            active = telop_mask & valid_frames

            if active.any():
                loss = F.smooth_l1_loss(pred_bbox[active], gt_bbox[active])
            else:
                loss = torch.tensor(0.0, device=device, requires_grad=True)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0.0, 0
    all_metrics = []

    with torch.no_grad():
        for batch in eval_loader:
            channel_ids = batch["channel_ids"].to(device)
            telop_feats = batch["telop_feats"].to(device)
            telop_mask  = batch["telop_mask"].to(device)
            time_feats  = batch["time_feats"].to(device)
            heatmaps    = batch["heatmaps"].to(device)
            gt_bbox     = batch["gt_bbox"].to(device)
            frame_mask  = batch["frame_mask"].to(device)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                pred_bbox = model(channel_ids, telop_feats, telop_mask,
                                  time_feats, heatmaps, frame_mask)

                valid_frames = frame_mask.unsqueeze(-1).expand_as(telop_mask)
                active = telop_mask & valid_frames

                if active.any():
                    loss = F.smooth_l1_loss(pred_bbox[active], gt_bbox[active])
                else:
                    loss = torch.tensor(0.0, device=device)

            eval_loss_sum += loss.item()
            eval_batches += 1

            m = compute_bbox_metrics(pred_bbox, gt_bbox, telop_mask, frame_mask)
            if m is not None:
                all_metrics.append(m)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss  = eval_loss_sum  / max(eval_batches, 1)
    lr_now = optimizer.param_groups[0]["lr"]

    if all_metrics:
        avg_m = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
    else:
        avg_m = {"mae_x": 0, "mae_y": 0, "mae_w": 0, "mae_h": 0, "mae_total": 0}

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"x={avg_m['mae_x']:.2f}  y={avg_m['mae_y']:.2f}  "
        f"w={avg_m['mae_w']:.2f}  h={avg_m['mae_h']:.2f}  "
        f"total={avg_m['mae_total']:.2f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[  1/100]  train=0.0196  eval=0.0187  x=9.55  y=18.16  w=15.71  h=1.55  total=44.97  lr=1.00e-04
   💾 best 갱신 (eval_loss=0.0187)


[  2/100]  train=0.0182  eval=0.0170  x=8.56  y=16.00  w=14.76  h=1.49  total=40.80  lr=9.99e-05
   💾 best 갱신 (eval_loss=0.0170)


[  3/100]  train=0.0175  eval=0.0167  x=8.58  y=15.89  w=14.39  h=1.53  total=40.39  lr=9.98e-05
   💾 best 갱신 (eval_loss=0.0167)


[  4/100]  train=0.0172  eval=0.0165  x=7.86  y=16.27  w=13.53  h=1.41  total=39.07  lr=9.96e-05
   💾 best 갱신 (eval_loss=0.0165)


[  5/100]  train=0.0169  eval=0.0160  x=7.95  y=15.72  w=13.73  h=1.42  total=38.83  lr=9.94e-05
   💾 best 갱신 (eval_loss=0.0160)


...

[ 96/100]  train=0.0154  eval=0.0151  x=7.35  y=14.90  w=13.22  h=1.33  total=36.79  lr=3.94e-07


[ 97/100]  train=0.0158  eval=0.0151  x=7.35  y=14.90  w=13.22  h=1.33  total=36.80  lr=2.22e-07


[ 98/100]  train=0.0156  eval=0.0151  x=7.35  y=14.90  w=13.22  h=1.33  total=36.80  lr=9.87e-08


[ 99/100]  train=0.0157  eval=0.0151  x=7.35  y=14.91  w=13.22  h=1.33  total=36.81  lr=2.47e-08


[100/100]  train=0.0157  eval=0.0151  x=7.35  y=14.91  w=13.22  h=1.33  total=36.81  lr=0.00e+00

✅ 완료. Best eval_loss=0.0120
